In [6]:
#POOLED PCA

In [27]:
#packages
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import glob

In [28]:
#COMBINE DATASETS OF ALL THE STABLECOINS
files = glob.glob("clean_data/*.parquet")

df_list = []

for file in files:
    temp = pd.read_parquet(file)
    coin_name = file.split("/")[-1].replace(".parquet", "")
    temp["coin"] = coin_name

    df_list.append(temp)

# Combine everything
df = pd.concat(df_list, ignore_index=True)

# Ensure datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,fear_greed_index,fear_greed_index_category,fed_funds_rate
0,USDT,0,0,0,0,0,0,0,2020-11-25 23:59:59,2020-11-25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USDT,0,0,0,0,0,0,0,2020-11-26 23:59:59,2020-11-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USDT,0,0,0,0,0,0,0,2020-11-27 23:59:59,2020-11-27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USDT,0,0,0,0,0,0,0,2020-11-28 23:59:59,2020-11-28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USDT,0,0,0,0,0,0,0,2020-11-29 23:59:59,2020-11-29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
#check columns
print(df.columns)

Index(['symbol', 'depeg', 'depeg_future_1d', 'depeg_future_3d',
       'depeg_future_5d', 'depeg_future_7d', 'depeg_future_14d',
       'depeg_future_30d', 'timestamp', 'timeOpen', 'timeClose', 'timeHigh',
       'timeLow', 'open', 'high', 'low', 'close', 'volume', 'marketCap',
       'circulatingSupply', 'coin', 'Unnamed: 0', 'percent_change_24h',
       'percent_change_7d', 'percent_change_30d', 'volume_percent_change_24h',
       'volume_percent_change_7d', 'volume_percent_change_30d',
       'market_cap_percent_change_24h', 'market_cap_percent_change_7d',
       'market_cap_percent_change_30d',
       'circulating_supply_percent_change_24h',
       'circulating_supply_percent_change_7d',
       'circulating_supply_percent_change_30d', 'realized_daily_volatility',
       'peg_error', 'abs_peg_error', 'price_deviation_5d',
       'price_deviation_30d', 'downward_price_deviation_5d',
       'downward_price_deviation_30d', 'fear_greed_index',
       'fear_greed_index_category', 'fed_fu

In [30]:
features = [
    'percent_change_24h', 'percent_change_7d', 'percent_change_30d',
    'volume_percent_change_24h', 'volume_percent_change_7d', 'volume_percent_change_30d',
    'market_cap_percent_change_24h', 'market_cap_percent_change_7d', 'market_cap_percent_change_30d',
    'circulating_supply_percent_change_24h', 'circulating_supply_percent_change_7d',
    'circulating_supply_percent_change_30d',
    'realized_daily_volatility',
    'fear_greed_index', 'fed_funds_rate'
]

# standardisation
coin_scalers = {}
normalized_parts_train = []
normalized_parts_test = []

for coin in train_df['symbol'].unique():
    train_mask = train_df['symbol'] == coin
    test_mask = test_df['symbol'] == coin

    cs = StandardScaler()
    normalized_parts_train.append(
        pd.DataFrame(
            cs.fit_transform(train_df.loc[train_mask, features]),
            index=train_df[train_mask].index,
            columns=features
        )
    )
    normalized_parts_test.append(
        pd.DataFrame(
            cs.transform(test_df.loc[test_mask, features]),
            index=test_df[test_mask].index,
            columns=features
        )
    )
    coin_scalers[coin] = cs

X_train_normalized = pd.concat(normalized_parts_train).loc[train_df.index]
X_test_normalized = pd.concat(normalized_parts_test).loc[test_df.index]

In [31]:
def clean_features(df, features):
    df = df.copy()
    # Replace inf/-inf with NaN first
    df[features] = df[features].replace([np.inf, -np.inf], np.nan)

    # Fill NaN per coin per feature (forward fill, then backward fill, then 0)
    df[features] = (
        df.groupby('symbol')[features]
        .transform(lambda x: x.ffill().bfill())
        .fillna(0)
    )
    return df

train_df = clean_features(train_df, features)
test_df = clean_features(test_df, features)

# Sanity check — should print nothing if clean
for col in features:
    bad = (~np.isfinite(train_df[col])).sum()
    if bad > 0:
        print(f"train {col}: {bad} bad values remaining")

for col in features:
    bad = (~np.isfinite(test_df[col])).sum()
    if bad > 0:
        print(f"test {col}: {bad} bad values remaining")

In [32]:
# PCA on training set
n_components = 5  # tune as needed
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_normalized)
X_test_pca = pca.transform(X_test_normalized)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative:", pca.explained_variance_ratio_.cumsum())

# Reconstruction error
X_train_recon = pca.inverse_transform(X_train_pca)
train_df['recon_error'] = np.mean((X_train_normalized.values - X_train_recon)**2, axis=1)

X_test_recon = pca.inverse_transform(X_test_pca)
test_df['recon_error'] = np.mean((X_test_normalized.values - X_test_recon)**2, axis=1)

# Per-coin thresholds 
THRESHOLD_QUANTILE = 0.99 # tune as required to check for anomalies

thresholds = {}
for coin, group in train_df.groupby('symbol'):
    thresholds[coin] = group['recon_error'].quantile(THRESHOLD_QUANTILE)

Explained variance ratio: [0.26971823 0.13851075 0.1203768  0.10042178 0.06534174]
Cumulative: [0.26971823 0.40822898 0.52860578 0.62902756 0.6943693 ]


In [33]:
results = []

for coin, group in test_df.groupby('symbol'):
    y_true = group['depeg_future_3d'].astype(bool)
    y_pred = group['anomaly']

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        auc = roc_auc_score(y_true, group['recon_error'])
    except ValueError:
        auc = np.nan  # only one class present

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    results.append({
        'coin': coin,
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
        'specificity': round(specificity, 4),
        'auc_roc': round(auc, 4) if not np.isnan(auc) else np.nan,
        'n_anomalies': y_pred.sum(),
        'n_true_depegs': y_true.sum(),
        'threshold': round(thresholds.get(coin, np.nan), 6)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

coin  precision  recall     f1  specificity  auc_roc  n_anomalies  n_true_depegs  threshold
 DAI     0.0857  1.0000 0.1579       0.9792   0.5753           35              3   1.980257
USDC     0.0189  1.0000 0.0370       0.8645   0.5956          212              4   2.925228
USDP     0.0842  0.4444 0.1416       0.8242   0.8186          285             54   1.805924
USDT     0.0143  1.0000 0.0282       0.8652   0.6044          210              3   3.663193
 UST     1.0000  0.9993 0.9996       1.0000   1.0000         1411           1412   5.551418
